# STEM Data Generation via Knowledge Distillation

Generate balanced STEM training data by distilling reasoning chains from DeepSeek-R1-Distill-Qwen-32B.

**Runtime**: Google Colab A100 (80GB VRAM) with 4-bit quantization

**Target**: ~55,000 examples across 5 STEM domains (math, physics, chemistry, CS, biology)

**Output format**: JSONL with fields `instruction`, `output`, `domain`, `type`, `difficulty`

Each domain includes ~50% calculation/verifiable problems and ~50% conceptual questions.
All prompts and outputs are in Russian, using `<think>` tags for chain-of-thought reasoning in ChatML format.

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets sentencepiece protobuf

In [ ]:
# Configuration
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/training_data"
QUANTIZATION = "4bit"  # For A100 80GB

# Domain targets (balanced STEM)
DOMAIN_TARGETS = {
    "math": 15000,      # algebra, calculus, geometry, number theory
    "physics": 12000,   # mechanics, thermodynamics, EM, optics
    "chemistry": 10000, # stoichiometry, organic, inorganic, solutions
    "cs": 10000,        # algorithms, data structures, complexity, code
    "biology": 8000,    # cell biology, genetics, ecology, anatomy
}

# Each domain: ~50% calculation/verifiable + ~50% conceptual
CALC_RATIO = 0.5

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME} with {QUANTIZATION} quantization...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.eval()

print(f"Model loaded. Memory allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# STEM prompt templates - all in Russian
# Each domain has calc_templates (calculation/verifiable) and conceptual_templates
# Templates use ChatML format with <think> tags for chain-of-thought

STEM_TEMPLATES = {
    "math": {
        "calc_templates": [
            {"system": "Ты опытный преподаватель математики. Сгенерируй задачу по алгебре и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй задачу по алгебре уровня {difficulty}: решение уравнений, неравенств или систем. Задача должна иметь числовой ответ."},
            {"system": "Ты опытный преподаватель математики. Сгенерируй задачу по математическому анализу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй задачу по математическому анализу уровня {difficulty}: вычисление производных, интегралов или пределов."},
            {"system": "Ты опытный преподаватель математики. Сгенерируй задачу по геометрии и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй вычислительную задачу по геометрии уровня {difficulty}: нахождение площадей, объёмов, углов или длин."},
            {"system": "Ты опытный преподаватель математики. Сгенерируй задачу по теории чисел и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй задачу по теории чисел уровня {difficulty}: делимость, НОД, простые числа, сравнения по модулю."},
            {"system": "Ты опытный преподаватель математики. Сгенерируй задачу по комбинаторике и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй вычислительную задачу по комбинаторике уровня {difficulty}: перестановки, сочетания, подсчёт."},
            {"system": "Ты опытный преподаватель математики. Сгенерируй задачу по тригонометрии и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй вычислительную задачу по тригонометрии уровня {difficulty}: тригонометрические уравнения, тождества, вычисление значений."},
            {"system": "Ты опытный преподаватель математики. Сгенерируй задачу по линейной алгебре и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй вычислительную задачу по линейной алгебре уровня {difficulty}: определители, собственные значения, системы линейных уравнений."},
        ],
        "conceptual_templates": [
            {"system": "Ты опытный преподаватель математики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по алгебре уровня {difficulty}: объяснение свойств, доказательство утверждения или анализ метода решения."},
            {"system": "Ты опытный преподаватель математики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по математическому анализу уровня {difficulty}: смысл производной, интеграла, непрерывность, сходимость рядов."},
            {"system": "Ты опытный преподаватель математики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по геометрии уровня {difficulty}: свойства фигур, теоремы, геометрические преобразования."},
            {"system": "Ты опытный преподаватель математики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по теории чисел уровня {difficulty}: свойства простых чисел, основная теорема арифметики, кольца вычетов."},
            {"system": "Ты опытный преподаватель математики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по теории вероятностей уровня {difficulty}: случайные события, распределения, центральная предельная теорема."},
            {"system": "Ты опытный преподаватель математики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по дискретной математике уровня {difficulty}: графы, логика, множества, отношения."},
            {"system": "Ты опытный преподаватель математики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Объясни связь между двумя математическими концепциями уровня {difficulty}. Почему одно понятие вытекает из другого?"},
        ],
    },
    "physics": {
        "calc_templates": [
            {"system": "Ты опытный преподаватель физики. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по механике уровня {difficulty}: кинематика, динамика, законы Ньютона, работа и энергия."},
            {"system": "Ты опытный преподаватель физики. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по термодинамике уровня {difficulty}: теплообмен, газовые законы, циклы, энтропия."},
            {"system": "Ты опытный преподаватель физики. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по электромагнетизму уровня {difficulty}: закон Кулона, цепи, индукция, магнитное поле."},
            {"system": "Ты опытный преподаватель физики. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по оптике уровня {difficulty}: преломление, дифракция, интерференция, линзы."},
            {"system": "Ты опытный преподаватель физики. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по колебаниям и волнам уровня {difficulty}: гармонические колебания, резонанс, волновое уравнение."},
            {"system": "Ты опытный преподаватель физики. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по ядерной физике уровня {difficulty}: радиоактивный распад, энергия связи, ядерные реакции."},
        ],
        "conceptual_templates": [
            {"system": "Ты опытный преподаватель физики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по механике уровня {difficulty}: объяснение физического явления, принципа или закона."},
            {"system": "Ты опытный преподаватель физики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по термодинамике уровня {difficulty}: объяснение процессов, начала термодинамики, тепловые машины."},
            {"system": "Ты опытный преподаватель физики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по электромагнетизму уровня {difficulty}: уравнения Максвелла, электромагнитные волны, принцип суперпозиции."},
            {"system": "Ты опытный преподаватель физики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по современной физике уровня {difficulty}: специальная теория относительности, квантовая механика, корпускулярно-волновой дуализм."},
            {"system": "Ты опытный преподаватель физики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по оптике уровня {difficulty}: природа света, голография, поляризация."},
            {"system": "Ты опытный преподаватель физики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Объясни физический парадокс или контринтуитивное явление уровня {difficulty}. Почему наивная интуиция ошибается?"},
        ],
    },
    "chemistry": {
        "calc_templates": [
            {"system": "Ты опытный преподаватель химии. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по стехиометрии уровня {difficulty}: расчёт по уравнениям реакций, выход продукта, избыток реагента."},
            {"system": "Ты опытный преподаватель химии. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по растворам уровня {difficulty}: концентрация, разбавление, смешивание, pH."},
            {"system": "Ты опытный преподаватель химии. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по термохимии уровня {difficulty}: тепловой эффект, закон Гесса, энтальпия."},
            {"system": "Ты опытный преподаватель химии. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по электрохимии уровня {difficulty}: ЭДС, уравнение Нернста, электролиз, законы Фарадея."},
            {"system": "Ты опытный преподаватель химии. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по химической кинетике уровня {difficulty}: скорость реакции, порядок реакции, константа скорости."},
        ],
        "conceptual_templates": [
            {"system": "Ты опытный преподаватель химии. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по органической химии уровня {difficulty}: механизмы реакций, функциональные группы, изомерия, номенклатура."},
            {"system": "Ты опытный преподаватель химии. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по неорганической химии уровня {difficulty}: строение атома, периодический закон, химическая связь, кристаллические решётки."},
            {"system": "Ты опытный преподаватель химии. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по химическому равновесию уровня {difficulty}: принцип Ле Шателье, константа равновесия, факторы смещения."},
            {"system": "Ты опытный преподаватель химии. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по коллоидной химии и растворам уровня {difficulty}: растворимость, коллигативные свойства, осмос."},
            {"system": "Ты опытный преподаватель химии. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по окислительно-восстановительным реакциям уровня {difficulty}: степени окисления, ОВР, электронный баланс."},
        ],
    },
    "cs": {
        "calc_templates": [
            {"system": "Ты опытный преподаватель информатики. Сгенерируй задачу на программирование и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ с кодом на Python.", "prompt": "Сгенерируй задачу на алгоритмы уровня {difficulty}: сортировка, поиск, жадные алгоритмы, бинарный поиск. Задача должна иметь конкретный ответ."},
            {"system": "Ты опытный преподаватель информатики. Сгенерируй задачу на программирование и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ с кодом на Python.", "prompt": "Сгенерируй задачу на структуры данных уровня {difficulty}: стеки, очереди, деревья, хеш-таблицы, графы. Задача с конкретным результатом."},
            {"system": "Ты опытный преподаватель информатики. Сгенерируй задачу на программирование и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ с кодом на Python.", "prompt": "Сгенерируй задачу на динамическое программирование уровня {difficulty}: подсчёт путей, оптимальные подструктуры, мемоизация."},
            {"system": "Ты опытный преподаватель информатики. Сгенерируй задачу на программирование и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ с кодом на Python.", "prompt": "Сгенерируй задачу на теорию графов уровня {difficulty}: обходы, кратчайшие пути, остовные деревья, потоки в сетях."},
            {"system": "Ты опытный преподаватель информатики. Сгенерируй задачу на программирование и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ с кодом на Python.", "prompt": "Сгенерируй задачу на рекурсию и комбинаторику уровня {difficulty}: генерация перестановок, подмножеств, рекуррентные соотношения."},
        ],
        "conceptual_templates": [
            {"system": "Ты опытный преподаватель информатики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по сложности алгоритмов уровня {difficulty}: O-нотация, классы P и NP, NP-полнота, редукции."},
            {"system": "Ты опытный преподаватель информатики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по архитектуре компьютера уровня {difficulty}: кэш, конвейер, виртуальная память, многопоточность."},
            {"system": "Ты опытный преподаватель информатики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по операционным системам уровня {difficulty}: процессы, потоки, планирование, взаимоблокировка."},
            {"system": "Ты опытный преподаватель информатики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по базам данных уровня {difficulty}: нормализация, SQL vs NoSQL, индексы, транзакции, ACID."},
            {"system": "Ты опытный преподаватель информатики. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по парадигмам программирования уровня {difficulty}: ООП, функциональное, паттерны проектирования, SOLID."},
        ],
    },
    "biology": {
        "calc_templates": [
            {"system": "Ты опытный преподаватель биологии. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по генетике уровня {difficulty}: законы Менделя, скрещивание, вероятности генотипов, задачи на наследование."},
            {"system": "Ты опытный преподаватель биологии. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по молекулярной биологии уровня {difficulty}: репликация ДНК, транскрипция, трансляция, подсчёт нуклеотидов и аминокислот."},
            {"system": "Ты опытный преподаватель биологии. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по экологии уровня {difficulty}: динамика популяций, цепи питания, продуктивность экосистемы, расчёт биомассы."},
            {"system": "Ты опытный преподаватель биологии. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по биохимии уровня {difficulty}: энергетический обмен, гликолиз, цикл Кребса, окислительное фосфорилирование, подсчёт АТФ."},
            {"system": "Ты опытный преподаватель биологии. Сгенерируй расчётную задачу и подробное решение. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй расчётную задачу по эволюции уровня {difficulty}: частоты аллелей, уравнение Харди-Вайнберга, отбор, дрейф генов."},
        ],
        "conceptual_templates": [
            {"system": "Ты опытный преподаватель биологии. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по клеточной биологии уровня {difficulty}: строение клетки, органеллы, мембранный транспорт, деление клетки."},
            {"system": "Ты опытный преподаватель биологии. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по генетике уровня {difficulty}: мутации, генная регуляция, эпигенетика, геномное редактирование."},
            {"system": "Ты опытный преподаватель биологии. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по экологии уровня {difficulty}: биоразнообразие, экосистемные услуги, сукцессия, глобальные экологические проблемы."},
            {"system": "Ты опытный преподаватель биологии. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по анатомии и физиологии человека уровня {difficulty}: нервная система, эндокринная система, иммунитет, кровообращение."},
            {"system": "Ты опытный преподаватель биологии. Сгенерируй концептуальный вопрос и подробный ответ. Сначала размышляй в тегах <think>, затем дай ответ.", "prompt": "Сгенерируй концептуальный вопрос по эволюционной биологии уровня {difficulty}: механизмы эволюции, видообразование, филогенетика, молекулярные часы."},
        ],
    },
}

DIFFICULTY_LEVELS = ["школьный", "базовый университетский", "продвинутый", "олимпиадный"]

# Verify template counts
for domain, templates in STEM_TEMPLATES.items():
    n_calc = len(templates["calc_templates"])
    n_concept = len(templates["conceptual_templates"])
    print(f"{domain}: {n_calc} calc + {n_concept} conceptual = {n_calc + n_concept} templates")

In [ ]:
import random
import json


def generate_examples(domain: str, question_type: str, batch_size: int = 4) -> list[dict]:
    """
    Generate STEM examples via knowledge distillation.

    Args:
        domain: One of 'math', 'physics', 'chemistry', 'cs', 'biology'
        question_type: 'calc' or 'conceptual'
        batch_size: Number of examples to generate in one call

    Returns:
        List of dicts with fields: instruction, output, domain, type, difficulty
    """
    templates = STEM_TEMPLATES[domain]
    template_key = f"{question_type}_templates"
    template_list = templates[template_key]

    results = []

    for _ in range(batch_size):
        # Select random template and difficulty
        template = random.choice(template_list)
        difficulty = random.choice(DIFFICULTY_LEVELS)

        # Build ChatML prompt with <think> tags
        system_msg = template["system"]
        user_msg = template["prompt"].format(difficulty=difficulty)

        chatml_prompt = (
            f"<|im_start|>system\n{system_msg}<|im_end|>\n"
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )

        # Tokenize
        inputs = tokenizer(chatml_prompt, return_tensors="pt").to(model.device)

        # Generate
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=2048,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode only the generated part
        generated_text = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=False,
        )

        # Clean up end tokens
        for stop_token in ["<|im_end|>", "<|endoftext|>"]:
            if stop_token in generated_text:
                generated_text = generated_text[:generated_text.index(stop_token)]
        generated_text = generated_text.strip()

        # Skip empty or too short generations
        if len(generated_text) < 50:
            continue

        results.append({
            "instruction": user_msg,
            "output": generated_text,
            "domain": domain,
            "type": question_type,
            "difficulty": difficulty,
        })

    return results


# Quick test
test_examples = generate_examples("math", "calc", batch_size=1)
if test_examples:
    print(f"Generated {len(test_examples)} example(s)")
    ex = test_examples[0]
    print(f"Domain: {ex['domain']}, Type: {ex['type']}, Difficulty: {ex['difficulty']}")
    print(f"Instruction: {ex['instruction'][:100]}...")
    print(f"Output length: {len(ex['output'])} chars")
    print(f"Has <think>: {'<think>' in ex['output']}")
else:
    print("No examples generated - check model output")

In [ ]:
import time
from tqdm.notebook import tqdm

OUTPUT_FILE = os.path.join(OUTPUT_DIR, "raw_stem.jsonl")
SAVE_EVERY = 1000
BATCH_SIZE = 4


def load_existing_progress(filepath: str) -> dict:
    """Load existing examples to support resume."""
    domain_counts = {d: {"calc": 0, "conceptual": 0} for d in DOMAIN_TARGETS}
    total = 0
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    ex = json.loads(line.strip())
                    d = ex.get("domain", "")
                    t = ex.get("type", "")
                    if d in domain_counts and t in domain_counts[d]:
                        domain_counts[d][t] += 1
                        total += 1
                except json.JSONDecodeError:
                    continue
    return domain_counts, total


def main_generation_loop():
    """Main loop: iterate over domains, generate calc + conceptual, save progress."""
    # Resume support
    domain_counts, existing_total = load_existing_progress(OUTPUT_FILE)
    if existing_total > 0:
        print(f"Resuming from {existing_total} existing examples")
        for d, counts in domain_counts.items():
            print(f"  {d}: {counts['calc']} calc + {counts['conceptual']} conceptual")

    # Calculate total target
    total_target = sum(DOMAIN_TARGETS.values())
    print(f"\nTotal target: {total_target} examples")
    print(f"Already generated: {existing_total}")
    print(f"Remaining: {total_target - existing_total}")

    # Progress bar
    pbar = tqdm(total=total_target, initial=existing_total, desc="Generating STEM data")

    buffer = []  # Buffer for batch saving
    errors = 0
    start_time = time.time()

    for domain, target_count in DOMAIN_TARGETS.items():
        calc_target = int(target_count * CALC_RATIO)
        conceptual_target = target_count - calc_target

        # Generate calculation examples
        while domain_counts[domain]["calc"] < calc_target:
            remaining = calc_target - domain_counts[domain]["calc"]
            batch = min(BATCH_SIZE, remaining)

            try:
                examples = generate_examples(domain, "calc", batch_size=batch)
                for ex in examples:
                    buffer.append(ex)
                    domain_counts[domain]["calc"] += 1
                    pbar.update(1)

                    # Save progress every SAVE_EVERY examples
                    if len(buffer) >= SAVE_EVERY:
                        with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
                            for item in buffer:
                                f.write(json.dumps(item, ensure_ascii=False) + "\n")
                        elapsed = time.time() - start_time
                        total_done = sum(c["calc"] + c["conceptual"] for c in domain_counts.values())
                        rate = total_done / (elapsed / 3600) if elapsed > 0 else 0
                        pbar.set_postfix({
                            "domain": domain,
                            "saved": total_done,
                            "rate": f"{rate:.0f}/hr",
                            "errors": errors,
                        })
                        buffer = []

            except Exception as e:
                errors += 1
                print(f"\nError [{domain}/calc]: {e}")
                if errors > 50:
                    print("Too many errors, stopping.")
                    break
                time.sleep(2)  # Brief pause on error
                continue

        # Generate conceptual examples
        while domain_counts[domain]["conceptual"] < conceptual_target:
            remaining = conceptual_target - domain_counts[domain]["conceptual"]
            batch = min(BATCH_SIZE, remaining)

            try:
                examples = generate_examples(domain, "conceptual", batch_size=batch)
                for ex in examples:
                    buffer.append(ex)
                    domain_counts[domain]["conceptual"] += 1
                    pbar.update(1)

                    if len(buffer) >= SAVE_EVERY:
                        with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
                            for item in buffer:
                                f.write(json.dumps(item, ensure_ascii=False) + "\n")
                        elapsed = time.time() - start_time
                        total_done = sum(c["calc"] + c["conceptual"] for c in domain_counts.values())
                        rate = total_done / (elapsed / 3600) if elapsed > 0 else 0
                        pbar.set_postfix({
                            "domain": domain,
                            "saved": total_done,
                            "rate": f"{rate:.0f}/hr",
                            "errors": errors,
                        })
                        buffer = []

            except Exception as e:
                errors += 1
                print(f"\nError [{domain}/conceptual]: {e}")
                if errors > 50:
                    print("Too many errors, stopping.")
                    break
                time.sleep(2)
                continue

        print(f"\n{domain} complete: {domain_counts[domain]}")

    # Flush remaining buffer
    if buffer:
        with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
            for item in buffer:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

    pbar.close()

    elapsed = time.time() - start_time
    total_done = sum(c["calc"] + c["conceptual"] for c in domain_counts.values())
    print(f"\nGeneration complete!")
    print(f"Total examples: {total_done}")
    print(f"Errors: {errors}")
    print(f"Time: {elapsed / 3600:.1f} hours")
    print(f"Rate: {total_done / (elapsed / 3600):.0f} examples/hour")
    print(f"Output: {OUTPUT_FILE}")


# Run the generation
main_generation_loop()

In [ ]:
# Quality check: load generated data and show statistics
import json
from collections import Counter

OUTPUT_FILE = os.path.join(OUTPUT_DIR, "raw_stem.jsonl")

examples = []
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        try:
            examples.append(json.loads(line.strip()))
        except json.JSONDecodeError:
            continue

print(f"Total examples loaded: {len(examples)}")
print(f"{'='*60}")

# Per-domain statistics
domain_counts = Counter(ex["domain"] for ex in examples)
type_counts = Counter(ex["type"] for ex in examples)
difficulty_counts = Counter(ex["difficulty"] for ex in examples)

print(f"\nDomain distribution:")
total = len(examples)
for domain in DOMAIN_TARGETS:
    count = domain_counts.get(domain, 0)
    pct = 100 * count / total if total > 0 else 0
    target = DOMAIN_TARGETS[domain]
    status = "OK" if count >= target * 0.95 else "LOW"
    print(f"  {domain:12s}: {count:6d} / {target:6d} ({pct:5.1f}%) [{status}]")

print(f"\nType distribution:")
for qtype, count in type_counts.most_common():
    print(f"  {qtype:15s}: {count:6d} ({100*count/total:.1f}%)")

print(f"\nDifficulty distribution:")
for diff, count in difficulty_counts.most_common():
    print(f"  {diff:25s}: {count:6d} ({100*count/total:.1f}%)")

# Per-domain type balance
print(f"\nPer-domain calc/conceptual balance:")
for domain in DOMAIN_TARGETS:
    domain_exs = [ex for ex in examples if ex["domain"] == domain]
    calc = sum(1 for ex in domain_exs if ex["type"] == "calc")
    concept = sum(1 for ex in domain_exs if ex["type"] == "conceptual")
    total_d = len(domain_exs)
    calc_pct = 100 * calc / total_d if total_d > 0 else 0
    print(f"  {domain:12s}: {calc} calc ({calc_pct:.0f}%) + {concept} conceptual ({100-calc_pct:.0f}%)")

# Balance check: no domain > 25%, each >= 12%
print(f"\nBalance check:")
balance_ok = True
for domain in DOMAIN_TARGETS:
    pct = 100 * domain_counts.get(domain, 0) / total if total > 0 else 0
    if pct > 25:
        print(f"  WARNING: {domain} is {pct:.1f}% (> 25% threshold)")
        balance_ok = False
    elif pct < 12:
        print(f"  WARNING: {domain} is {pct:.1f}% (< 12% threshold)")
        balance_ok = False

if balance_ok:
    print("  All domains within balance thresholds (12%-25%)")

# Output length statistics
output_lengths = [len(ex["output"]) for ex in examples]
has_think = sum(1 for ex in examples if "<think>" in ex["output"])
print(f"\nOutput statistics:")
print(f"  Avg length: {sum(output_lengths)/len(output_lengths):.0f} chars")
print(f"  Min length: {min(output_lengths)} chars")
print(f"  Max length: {max(output_lengths)} chars")
print(f"  With <think> tags: {has_think}/{len(examples)} ({100*has_think/len(examples):.1f}%)")

## Next Steps

1. **Verify answers**: Run `verify_answers.py` to check calculation problems against ground truth (SymPy, numerical checks)
2. **Sort curriculum**: Run `sort_curriculum.py` to order examples by difficulty and topic for curriculum-aware SFT
3. **Proceed to SFT**: Use the verified and sorted dataset for supervised fine-tuning with QLoRA on the target model